In [1]:
import pandas as pd
import numpy as np

# Load the cleaned data
matches = pd.read_csv('../data/processed/matches_filtered.csv')
teams_df = pd.read_csv('../data/processed/wc2026_teams.csv')
teams = teams_df['team'].tolist()

print(f"Loaded {len(matches)} matches for {len(teams)} teams")
print(f"Date range: {matches['date'].min()} to {matches['date'].max()}")
print(f"Tournaments: {matches['tournament'].unique()[:5]}")

Loaded 1505 matches for 48 teams
Date range: 2014-01-07 to 2026-06-27
Tournaments: <StringArray>
[                   'WAFF Championship',
                             'Friendly',
                       'FIFA World Cup',
                  'Kirin Challenge Cup',
 'African Cup of Nations qualification']
Length: 5, dtype: str


In [2]:
# Define K-factors by tournament type
def get_k_factor(tournament):
    """Assign K-factor based on match importance"""
    if 'World Cup' in tournament and 'qualification' not in tournament:
        return 60
    elif 'World Cup qualification' in tournament:
        return 40
    elif any(x in tournament for x in ['Euro', 'Copa', 'African Cup', 'Asian Cup', 'Gold Cup']):
        return 50
    else:
        return 20  # Friendlies and others

# Test the K-factor mapping
print("K-factor samples:")
for tournament in matches['tournament'].unique()[:5]:
    print(f"  {tournament}: {get_k_factor(tournament)}")

# Initialize Elo ratings
elo_ratings = {team: 1500 for team in teams}
print(f"\nInitialized {len(elo_ratings)} teams at 1500 rating")

# Sort matches by date to process chronologically
matches_sorted = matches.sort_values('date').reset_index(drop=True)

# Process each match
for idx, row in matches_sorted.iterrows():
    home_team = row['home_team']
    away_team = row['away_team']
    home_goals = row['home_score']
    away_goals = row['away_score']
    
    # Get current ratings
    home_rating = elo_ratings[home_team]
    away_rating = elo_ratings[away_team]
    
    # Calculate expected score
    expected_home = 1 / (1 + 10 ** ((away_rating - home_rating) / 400))
    expected_away = 1 - expected_home
    
    # Determine actual score (1=win, 0.5=draw, 0=loss)
    if home_goals > away_goals:
        actual_home = 1
        actual_away = 0
    elif home_goals == away_goals:
        actual_home = 0.5
        actual_away = 0.5
    else:
        actual_home = 0
        actual_away = 1
    
    # Get K-factor
    k = get_k_factor(row['tournament'])
    
    # Update ratings
    elo_ratings[home_team] = home_rating + k * (actual_home - expected_home)
    elo_ratings[away_team] = away_rating + k * (actual_away - expected_away)

print(f"\nProcessed {len(matches_sorted)} matches")
print(f"\nTop 10 teams by Elo rating:")
sorted_teams = sorted(elo_ratings.items(), key=lambda x: x[1], reverse=True)
for team, rating in sorted_teams[:10]:
    print(f"  {team:25s} {rating:7.1f}")

K-factor samples:
  WAFF Championship: 20
  Friendly: 20
  FIFA World Cup: 60
  Kirin Challenge Cup: 20
  African Cup of Nations qualification: 50

Initialized 48 teams at 1500 rating

Processed 1505 matches

Top 10 teams by Elo rating:
  France                     1621.2
  Spain                      1616.0
  Argentina                  1611.2
  Portugal                   1600.1
  Austria                    1583.6
  Algeria                    1582.6
  Brazil                     1572.9
  Morocco                    1571.7
  Croatia                    1570.6
  Japan                      1568.7
